In [ ]:
%matplotlib tk
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from torch.nn.functional import conv2d, conv3d
from scipy.ndimage import convolve, generate_binary_structure
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def wave_equation_2d(psi_0, psi_1, dt, dx, dy, c, mask=None):
    """
    Simulates the 2D wave equation using explicit finite differences with Dirichlet (zero) boundary conditions inside a mask.
    mask: boolean tensor, True inside the domain, False outside
    """
    c2 = c[1:-1,1:-1] ** 2
    NX, NY = psi_0.shape
    psi_2 = torch.zeros_like(psi_0)
    # Interior points (finite difference Laplacian)
    psi_2[1:-1,1:-1] = (
        2 * psi_1[1:-1,1:-1] - psi_0[1:-1,1:-1]
        + c2 * dt**2 * (
            (psi_1[2:,1:-1] - 2*psi_1[1:-1,1:-1] + psi_1[:-2,1:-1]) / dx**2
            + (psi_1[1:-1,2:] - 2*psi_1[1:-1,1:-1] + psi_1[1:-1,:-2]) / dy**2
        )
    )
    if mask is not None:
        # Dirichlet boundary: set all points outside mask to zero
        psi_2[~mask] = 0
        # Set boundary points (where mask is True but at least one neighbor is False) to zero
        # Top
        psi_2[0, mask[0,:]] = 0
        # Bottom
        psi_2[-1, mask[-1,:]] = 0
        # Left
        psi_2[mask[:,0], 0] = 0
        # Right
        psi_2[mask[:,-1], -1] = 0
    else:
        # Standard Dirichlet boundaries (whole grid)
        psi_2[0, :] = 0
        psi_2[-1, :] = 0
        psi_2[:, 0] = 0
        psi_2[:, -1] = 0
    return psi_2

In [ ]:
# Efficient animation: update psi in-place and avoid storing all iterations

NX, NY = 256, 256
n_iterations = 1000
dx = 0.1
dy = 0.1
c0 = 1.0  # background speed
c_lens = 0.5  # slower speed in lens
courant_number = 0.2
dt = courant_number * min(dx, dy)**2 / c0
x, y = torch.meshgrid(torch.linspace(0, NX-1, NX), torch.linspace(0, NY-1, NY), indexing='ij')

# Ellipse parameters
center_x, center_y = NX // 2, NY // 2
ellipse_a = NX // 2.5  # semi-major axis (x-direction)
ellipse_b = NY // 3.5  # semi-minor axis (y-direction)
ellipse_mask = (((x - center_x) / ellipse_a) ** 2 + ((y - center_y) / ellipse_b) ** 2) <= 1
ellipse_mask = torch.tensor(ellipse_mask, dtype=torch.bool, device=device)

# Parabola Parameters
vertex_x, vertex_y = NX // 4, NY // 2
focus = NX // 16
focus_x, focus_y = vertex_x + focus, vertex_y
parabola_mask = 4*focus* x < (y - vertex_y)**2
parabola_mask = torch.tensor(parabola_mask, dtype=torch.bool, device=device)

# Calculate left focus of the ellipse
c_ellipse = np.sqrt(np.abs(ellipse_a**2 - ellipse_b**2))
left_focus_x = center_x - c_ellipse
left_focus_y = center_y

# Place initial wave at the left focus
psi_0 = 8*torch.exp(-((x - focus_x + NX//4)**2 + (y - focus_y)**2) / (2 * (NX/80)**2)).to(device)
psi_1 = psi_0.clone()

# Define a lens-shaped region as the intersection of two spheres
sphere_radius = NX // 4.5
sphere1_center = (center_x - NX // 5.5, center_y)
sphere2_center = (center_x + NX // 5.5, center_y)
sphere1 = ((x - sphere1_center[0])**2 + (y - sphere1_center[1])**2) <= sphere_radius**2
sphere2 = ((x - sphere2_center[0])**2 + (y - sphere2_center[1])**2) <= sphere_radius**2
lens_mask = sphere1 & sphere2

# Create spatially varying wave speed
c = torch.full((NX, NY), c0, device=device)
c[lens_mask] = c_lens

# --- Single slit mask ---
# Block all except a vertical slit between source and lens
mask = torch.ones((NX, NY), dtype=torch.bool, device=device)
slit_x = center_x #- NX // 6  # x-position of slit (between source and lens)
slit_width = NX // 100       # width of the slit
slit_y0 = center_y - NY // 8
slit_y1 = center_y + NY // 8
mask[:, :] = False  # Block all
mask[slit_x-slit_width//2:slit_x+slit_width//2, slit_y0:slit_y1] = True  # Open slit
mask[x >= slit_x + slit_width//2] = True  # Open everything to the right of the slit
mask[x < slit_x - slit_width//2] = True  # Open everything to the left of the slit
print(mask.device, parabola_mask.device)
mask ^= parabola_mask

# Visualize the mask and lens

plt.imshow(mask.cpu().numpy().T, cmap='gray', alpha=0.5, extent=(0, NX*dx, 0, NY*dy))
plt.contour(lens_mask.cpu().numpy().T, levels=[0.5], colors='red', linewidths=1, extent=(0, NX*dx, 0, NY*dy))
plt.title('Single Slit Mask and Lens')


cuda:0 cuda:0


C:\Users\hasan\AppData\Local\Temp\ipykernel_49628\308025706.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ellipse_mask = torch.tensor(ellipse_mask, dtype=torch.bool, device=device)
C:\Users\hasan\AppData\Local\Temp\ipykernel_49628\308025706.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  parabola_mask = torch.tensor(parabola_mask, dtype=torch.bool, device=device)


Text(0.5, 1.0, 'Single Slit Mask and Lens')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_facecolor('white')
im = ax.imshow(psi_0.cpu().numpy().T, cmap='seismic', extent=(0, NX*dx, 0, NY*dy), vmax=1, vmin=-1)
contour = ax.contour(lens_mask.cpu().numpy().T, levels=[0.5], colors='black', linewidths=1, extent=(0, NX*dx, 0, NY*dy))
slit_overlay = ax.contour(mask.cpu().numpy().T, levels=[0.5], colors='red', linewidths=1, extent=(0, NX*dx, 0, NY*dy))
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)
t = np.linspace(0, n_iterations * dt, n_iterations + 1)

# Only keep two arrays in memory, update in-place
psi_prev = psi_0.clone()
psi_curr = psi_1.clone()
import gc

source_amplitude = 0.00001  # Amplitude of the source term
psi_source = source_amplitude*torch.exp(-((x - center_x + NX//4)**2 + (y - center_y)**2) / (2 * (NX/100)**2)).to(device)

frames_to_advance = 50  # Number of simulation steps per animation frame

def update(frame):
    global psi_prev, psi_curr, contour, slit_overlay
    for i in range(frames_to_advance):
        t_global = (frame * frames_to_advance + i) * dt
        source_term = psi_source * np.sin(2 * np.pi * t_global / (dt * frames_to_advance * 50))*dt
        psi_next = wave_equation_2d(psi_prev, psi_curr, dt, dx, dy, c, mask=mask) + source_term
        psi_prev = psi_curr.clone()
        psi_curr = psi_next.clone()
    im.set_array(psi_curr.cpu().numpy().T)
    time_text.set_text(f'Time = {frame*frames_to_advance*dt:.2f} s')
    # Remove old contour and draw new one
    for coll in contour.collections:
        coll.remove()
    for coll in slit_overlay.collections:
        coll.remove()
    contour = ax.contour(lens_mask.cpu().numpy().T, levels=[0.5], colors='black', linewidths=1, extent=(0, NX*dx, 0, NY*dy))
    slit_overlay = ax.contour(mask.cpu().numpy().T, levels=[0.5], colors='red', linewidths=1, extent=(0, NX*dx, 0, NY*dy))
    return [im, time_text] + contour.collections + slit_overlay.collections

ani = animation.FuncAnimation(fig, update, frames=n_iterations, interval=5, blit=True)
plt.colorbar(im, ax=ax, label='Amplitude')
plt.xlabel('X')
plt.ylabel('Y')
plt.show()
#ani.save('/Users/hasan/Python Animations/wave_equation_animation_slit_lens.gif', writer='pillow', fps=50,dpi=200)